# Dental Implant Survival Analysis – Andersen–Gill Cox Models

**Target journal:** Journal of Clinical Periodontology (JCP)

This notebook performs a comprehensive survival analysis of dental implants using
Andersen–Gill (AG) Cox proportional hazards models with robust standard errors
(clustered by patient). Three models are fitted:

1. **Overall AG model** – full cohort
2. **Early-failure AG model** – failures within ≤365 days
3. **Late-failure AG model** – failures after >365 days

The analysis also includes Kaplan–Meier curves with time-at-risk tables,
expanded univariable survival screening, and a global proportional-hazards
diagnostic for each AG model.

All functions are imported from `functions.py` for modularity and reproducibility.

In [ ]:
# ============================================================
# 0. Setup & Imports
# ============================================================
import importlib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project module
import functions as fn
importlib.reload(fn)

fn.print_versions()

# Display settings
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Configuration

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
INPUT_PATH = r"C:\Users\cahana_s\research\research_shetel_2_3\shetel_2_3_data\implants_with_restoration_summary_all.xlsx"
OUTDIR     = r"C:\Users\cahana_s\research\research_shetel_2_3\outputs"
CENSOR_DATE   = "2025-08-24"
COHORT_CUTOFF = "2023-07-01"
INCIDENCE_CUTOFF_DATE = "2023-12-31"  # cutoff used in crude incidence tables/figure
EARLY_THRESHOLD_DAYS = 365   # ≤365 = early, >365 = late

import os
os.makedirs(OUTDIR, exist_ok=True)

## 2. Data Loading & Preprocessing

In [ ]:
# ============================================================
# 2. Data Loading & Preprocessing
# ============================================================
from pathlib import Path
import pandas as pd

PARQUET_PATH = Path("final_cohort.parquet")

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
else:
    df_raw = fn.load_data(INPUT_PATH)
    df = fn.preprocess(
        df_raw,
        censor_date=CENSOR_DATE,
        cohort_cutoff=COHORT_CUTOFF
    )

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    print("Casting object columns to string:", object_cols)

    for col in object_cols:
        df[col] = df[col].astype("string").str.strip()

    df.to_parquet(PARQUET_PATH, index=False)

print(f"Final cohort: {len(df):,} implants | {df['patient_id'].nunique():,} patients")
print(f"Failures:     {int(df['is_failure'].sum()):,} ({df['is_failure'].mean()*100:.1f}%)")

## 3. Descriptive Statistics

In [ ]:
# ============================================================
# 3a. Continuous variables
# ============================================================
continuous_cols = ["age_years", "length", "diameter", "days_to_failure"]
display(fn.style_table(
    fn.continuous_summary(df, continuous_cols),
    caption="Continuous Variables"
))

In [ ]:
# ============================================================
# 3b. Categorical / binary variables
# ============================================================
cat_cols = [
    "jaw", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv_lbl",
    "rehabilitation_type", "fixed_loading_type",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    "has_mahash_onimplantday", "has_resm_onimplantday",
    "has_resp_onorbeforeimplant",
]
display(fn.style_table(
    fn.frequency_table(df, cat_cols),
    caption="Categorical and Binary Variables"
))

In [ ]:
# ============================================================
# 3c. Follow-up summary
# ============================================================
display(fn.style_table(
    fn.follow_up_summary(df),
    caption="Follow-up Summary"
))

## 4. Implant Survival by Procedure Sequence

In [ ]:
# ============================================================
# 4. Stacked bar chart – survival vs failure by implant sequence
# ============================================================
fig = fn.plot_survival_by_sequence(df, save_path=os.path.join(OUTDIR, "survival_by_sequence"))
plt.show()

## 5. Kaplan-Meier Survival and Follow-up Completeness

The two panels are displayed side by side. Panel A shows potential follow-up using reverse Kaplan-Meier, with group medians marked directly on the plot. The time-at-risk table for panel B is printed separately below the figure.

**Panel A:** reverse Kaplan-Meier follow-up by implant sequence (1, 2, 3+).

**Panel B:** Kaplan-Meier survival by implant sequence (1, 2, 3+).

For panel A, follow-up is censored administratively at 2023-12-31.

In [ ]:
# ============================================================
# 5. Composite figure: panel A reverse KM; panel B KM
# ============================================================
df["years_to_failure"] = df["days_to_failure"] / 365.25
df["implant_group"] = df["implant_num_surv"].apply(lambda g: "3+" if g >= 3 else str(int(g)))

label_map = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

fig = fn.plot_implant_sequence_survival_followup_figure(
    df,
    save_path=os.path.join(OUTDIR, "km_reverse_km_implant_sequence"),
    study_end="2023-12-31",
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map,
    color_map=color_map,
)
plt.show()

km_risk_table = fn.km_time_at_risk_table(
    df,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map,
    x_cap=10.0,
)
print("Time at risk for panel B:")
display(fn.style_table(km_risk_table))

reverse_km_summary = fn.reverse_km_followup_summary(
    df,
    study_end="2023-12-31",
    group_col="implant_group",
    event_col="is_failure",
    label_map=label_map,
)
print("Potential follow-up summary for panel A:")
display(fn.style_table(reverse_km_summary))

In [ ]:
# ============================================================
# 6. Empirical check: follow-up starts at each site's own implant date
# ============================================================
study_end_panel_a = pd.Timestamp("2023-12-31")
study_end_panel_b = pd.Timestamp(CENSOR_DATE)

check = df.copy()
check["patient_id"] = check["patient_id"].astype("string")
check["tooth_number"] = pd.to_numeric(check["tooth_number"], errors="coerce")
check["site_id"] = (
    check["patient_id"].fillna("NA")
    + "_"
    + check["tooth_number"].astype("Int64").astype("string").fillna("NA")
)

check["duedate"] = pd.to_datetime(check["duedate"], errors="coerce")
check["failure_date"] = pd.to_datetime(check["failure_date"], errors="coerce")
check["is_failure"] = pd.to_numeric(check["is_failure"], errors="coerce").fillna(0).astype(int)
check["implant_index"] = pd.to_numeric(check["implant_index"], errors="coerce")
check["implant_num_surv"] = pd.to_numeric(check["implant_num_surv"], errors="coerce")

check["panel_a_end"] = check["failure_date"].where(check["is_failure"].eq(1), study_end_panel_a)
check["panel_b_end"] = check["failure_date"].where(check["is_failure"].eq(1), study_end_panel_b)

check["panel_a_followup_days"] = (check["panel_a_end"] - check["duedate"]).dt.days
check["panel_b_followup_days"] = (check["panel_b_end"] - check["duedate"]).dt.days

sites_3plus = (
    check.groupby("site_id")["implant_index"]
    .max()
    .loc[lambda s: s >= 3]
    .index
)

subset = (
    check.loc[check["site_id"].isin(sites_3plus), [
        "site_id",
        "patient_id",
        "tooth_number",
        "implant_index",
        "implant_num_surv",
        "duedate",
        "failure_date",
        "is_failure",
        "panel_a_end",
        "panel_a_followup_days",
        "panel_b_end",
        "panel_b_followup_days",
    ]]
    .sort_values(["site_id", "implant_index", "duedate"])
    .reset_index(drop=True)
)

summary_check = pd.DataFrame([{
    "Sites with >=3 implants": subset["site_id"].nunique(),
    "Patients represented": subset["patient_id"].nunique(),
    "Rows with implant_index == 2": int((subset["implant_index"] == 2).sum()),
    "Rows with implant_index >= 3": int((subset["implant_index"] >= 3).sum()),
    "All implant 2 rows have own start date": subset.loc[subset["implant_index"] == 2, "duedate"].notna().all(),
    "All implant 3+ rows have own start date": subset.loc[subset["implant_index"] >= 3, "duedate"].notna().all(),
}])

print("Empirical check by site_id (same patient + same tooth):")
display(fn.style_table(summary_check))

print("Example rows for sites with 3+ implants:")
display(fn.style_table(subset.head(50)))

In [ ]:
# ============================================================
# 6b. Composite figure: reverse KM excludes early failures; KM keeps full data
# ============================================================
early_failure_mask = (
    pd.to_numeric(df["is_failure"], errors="coerce").fillna(0).astype(int).eq(1)
    & pd.to_numeric(df["days_to_failure"], errors="coerce").notna()
    & (pd.to_numeric(df["days_to_failure"], errors="coerce") <= EARLY_THRESHOLD_DAYS)
)

df_reverse_no_early = df.loc[~early_failure_mask].copy()
df_reverse_no_early["years_to_failure"] = df_reverse_no_early["days_to_failure"] / 365.25
df_reverse_no_early["implant_group"] = df_reverse_no_early["implant_num_surv"].apply(
    lambda g: "3+" if g >= 3 else str(int(g))
)

df_km_full = df.copy()
df_km_full["years_to_failure"] = df_km_full["days_to_failure"] / 365.25
df_km_full["implant_group"] = df_km_full["implant_num_surv"].apply(
    lambda g: "3+" if g >= 3 else str(int(g))
)

label_map_early = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map_early = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

print(
    f"Reverse KM subset after excluding early failures (<= {EARLY_THRESHOLD_DAYS} days): "
    f"{len(df_reverse_no_early):,} implants"
)
print(f"Regular KM subset (full data): {len(df_km_full):,} implants")

fn._apply_jcp_style()
fig_early_failure = plt.figure(figsize=(13.2, 5.4), constrained_layout=True)
gs = fig_early_failure.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[1.1, 1.0],
    wspace=0.18,
)

ax_a = fig_early_failure.add_subplot(gs[0, 0])
ax_b = fig_early_failure.add_subplot(gs[0, 1])

combined_reverse = fn.compute_followup_time(
    df_reverse_no_early,
    study_end="2023-12-31",
    event_col="is_failure",
)
reverse_summary_early = fn.reverse_km_followup_summary(
    df_reverse_no_early,
    study_end="2023-12-31",
    group_col="implant_group",
    event_col="is_failure",
    label_map=label_map_early,
)

fn._plot_survival_curves_on_axis(
    combined_reverse,
    ax=ax_a,
    duration_col="followup_years",
    event_col="reverse_km_event",
    group_col="implant_group",
    label_map=label_map_early,
    color_map=color_map_early,
    title="Reverse Kaplan-Meier follow-up by implant sequence\n(excluding failures <= 1 year)",
    ylabel="Probability of remaining under follow-up",
    reverse=True,
    x_cap=10.0,
)
fn._annotate_reverse_km_medians(ax_a, reverse_summary_early, label_map_early, color_map_early)
ax_a.text(
    -0.12,
    1.04,
    "A",
    transform=ax_a.transAxes,
    ha="left",
    va="top",
    fontsize=12,
    fontweight="bold",
    color=fn.FIG_DEFAULTS["title_color"],
    clip_on=False,
)

fn._plot_survival_curves_on_axis(
    df_km_full,
    ax=ax_b,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map_early,
    color_map=color_map_early,
    title="Kaplan-Meier survival by implant sequence",
    ylabel="Survival probability",
    reverse=False,
    x_cap=10.0,
)
ax_b.text(
    -0.12,
    1.04,
    "B",
    transform=ax_b.transAxes,
    ha="left",
    va="top",
    fontsize=12,
    fontweight="bold",
    color=fn.FIG_DEFAULTS["title_color"],
    clip_on=False,
)

save_path_early = os.path.join(
    OUTDIR,
    "km_reverse_km_implant_sequence_reverse_excluding_early_failures",
)
for ext in ("png", "pdf"):
    fig_early_failure.savefig(
        f"{save_path_early}.{ext}",
        dpi=fn.FIG_DEFAULTS["dpi"],
        bbox_inches="tight",
    )
print(f"  Saved: {save_path_early}.png / .pdf")
plt.show()

km_risk_table_early = fn.km_time_at_risk_table(
    df_km_full,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map_early,
    x_cap=10.0,
)
print("Time at risk for panel B (full data):")
display(fn.style_table(km_risk_table_early))

print("Potential follow-up summary for panel A (excluding failures <= 1 year):")
display(fn.style_table(reverse_summary_early))

In [ ]:
# ============================================================
# 6c. Follow-up time distribution by implant index group
# ============================================================
study_end_followup = "2023-12-31"
followup_plot = fn.compute_followup_time(df, study_end=study_end_followup, event_col="is_failure")

group_source_col = "implant_index" if "implant_index" in followup_plot.columns else "implant_num_surv"
group_num = pd.to_numeric(followup_plot[group_source_col], errors="coerce")
followup_plot["implant_group"] = pd.Series(pd.NA, index=followup_plot.index, dtype="string")
followup_plot.loc[group_num.eq(1), "implant_group"] = "1"
followup_plot.loc[group_num.eq(2), "implant_group"] = "2"
followup_plot.loc[group_num.ge(3), "implant_group"] = "3+"
followup_plot = followup_plot.loc[followup_plot["implant_group"].isin(["1", "2", "3+"])].copy()

followup_bin_summary = fn.summarize_followup_bins(
    followup_plot,
    group_col="implant_group",
    followup_col="followup_years",
    allowed_groups=["1", "2", "3+"],
)

label_map = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

fn._apply_jcp_style()
fig_followup_dist, ax_followup_dist = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
fn.plot_followup_bin_summary(
    followup_bin_summary,
    ax=ax_followup_dist,
    label_map=label_map,
    color_map=color_map,
    title="Follow-up time distribution by implant index group",
)
ax_followup_dist.set_xlabel("Follow-up time category")

save_path_followup_dist = os.path.join(OUTDIR, "followup_time_distribution_by_implant_index_group")
for ext in ("png", "pdf"):
    fig_followup_dist.savefig(f"{save_path_followup_dist}.{ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
print(f"  Saved: {save_path_followup_dist}.png / .pdf")
plt.show()

In [ ]:
# ============================================================
# 6d. Follow-up time distribution by implant index group (percent)
# ============================================================
percent_bin_order = ["<=1 year", ">1-3 years", ">3-5 years", ">5 years"]
percent_group_order = ["1", "2", "3+"]

followup_percent_plot = (
    followup_bin_summary
    .assign(
        implant_group=followup_bin_summary["implant_group"].astype(str),
        followup_bin=followup_bin_summary["followup_bin"].astype(str),
    )
    .pivot(index="implant_group", columns="followup_bin", values="percent")
    .reindex(index=percent_group_order, columns=percent_bin_order)
    .fillna(0.0)
)

percent_colors = {
    "<=1 year": "#C44E52",
    ">1-3 years": "#DD8452",
    ">3-5 years": "#55A868",
    ">5 years": "#4C72B0",
}
percent_labels = {
    "1": "First implant",
    "2": "Second implant",
    "3+": "Third or more",
}

fn._apply_jcp_style()
fig_followup_pct, ax_followup_pct = plt.subplots(figsize=(8.6, 5.0), constrained_layout=True)

bottom = np.zeros(len(followup_percent_plot.index), dtype=float)
x = np.arange(len(followup_percent_plot.index))

for bin_name in percent_bin_order:
    values = followup_percent_plot[bin_name].to_numpy(dtype=float)
    bars = ax_followup_pct.bar(
        x,
        values,
        bottom=bottom,
        color=percent_colors[bin_name],
        edgecolor="white",
        linewidth=0.8,
        label=bin_name,
    )
    for bar, value, base in zip(bars, values, bottom):
        if value >= 6:
            ax_followup_pct.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=8,
                fontweight="bold",
            )
    bottom = bottom + values

ax_followup_pct.set_xticks(x)
ax_followup_pct.set_xticklabels([percent_labels.get(idx, idx) for idx in followup_percent_plot.index])
ax_followup_pct.set_ylabel("Percent of implants")
ax_followup_pct.set_xlabel("Implant index group")
ax_followup_pct.set_ylim(0, 100)
ax_followup_pct.set_title("Follow-up time distribution by implant index group (%)", color=fn.FIG_DEFAULTS["title_color"], pad=10)
ax_followup_pct.grid(axis="y", color=fn.FIG_DEFAULTS["grid_color"], alpha=0.6)
ax_followup_pct.legend(
    title="Follow-up time",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    ncol=1,
    framealpha=0.95,
    facecolor="white",
    edgecolor=fn.FIG_DEFAULTS["border_color"],
)
for spine in ["top", "right"]:
    ax_followup_pct.spines[spine].set_visible(False)

save_path_followup_pct = os.path.join(OUTDIR, "followup_time_distribution_by_implant_index_group_percent")
for ext in ("png", "pdf"):
    fig_followup_pct.savefig(f"{save_path_followup_pct}.{ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
print(f"  Saved: {save_path_followup_pct}.png / .pdf")
plt.show()

In [ ]:
# ============================================================
# 6e. Third-or-more implants performed per calendar year
# ============================================================
source_col = "implant_index" if "implant_index" in df.columns else "implant_num_surv"
implant_num = pd.to_numeric(df[source_col], errors="coerce")

df_3plus_years = df.loc[implant_num.ge(3)].copy()
df_3plus_years["performed_date"] = pd.to_datetime(df_3plus_years["duedate"], errors="coerce")
df_3plus_years = df_3plus_years.loc[df_3plus_years["performed_date"].notna()].copy()
df_3plus_years["performed_year"] = df_3plus_years["performed_date"].dt.year.astype(int)

yearly_counts = (
    df_3plus_years
    .groupby("performed_year", as_index=False)
    .size()
    .rename(columns={"size": "N"})
    .sort_values("performed_year")
    .reset_index(drop=True)
)

fn._apply_jcp_style()
fig_3plus_years, ax_3plus_years = plt.subplots(figsize=(9.2, 4.8), constrained_layout=True)

x = np.arange(len(yearly_counts))
bars = ax_3plus_years.bar(
    x,
    yearly_counts["N"],
    color=fn.WONG["green"],
    edgecolor="white",
    linewidth=0.8,
)

for bar, n_val in zip(bars, yearly_counts["N"]):
    ax_3plus_years.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(1, int(yearly_counts["N"].max() * 0.015)),
        f"{int(n_val):,}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=fn.FIG_DEFAULTS["title_color"],
    )

ax_3plus_years.set_xticks(x)
ax_3plus_years.set_xticklabels(yearly_counts["performed_year"].astype(str).tolist(), rotation=45, ha="right")
ax_3plus_years.set_ylabel("Number of implants")
ax_3plus_years.set_xlabel("Calendar year of implant performance")
ax_3plus_years.set_title("Third-or-more implants performed per calendar year", color=fn.FIG_DEFAULTS["title_color"], pad=10)
ax_3plus_years.grid(axis="y", color=fn.FIG_DEFAULTS["grid_color"], alpha=0.6)
for spine in ["top", "right"]:
    ax_3plus_years.spines[spine].set_visible(False)

save_path_3plus_years = os.path.join(OUTDIR, "third_or_more_implants_per_year")
try:
    for ext in ("png", "pdf"):
        fig_3plus_years.savefig(f"{save_path_3plus_years}.{ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
    print(f"  Saved: {save_path_3plus_years}.png / .pdf")
except KeyboardInterrupt:
    print("  Figure display is ready; file save was interrupted.")

display(fn.style_table(yearly_counts.rename(columns={"performed_year": "Year"})))
plt.show()

## 6. Univariable Survival Screening

In [ ]:
# ============================================================
# 6. Log-rank plus univariable Cox screening
# ============================================================
test_vars = [
    "jaw", "region", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv",
    "rehabilitation_type", "fixed_loading_type",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    "has_mahash_onimplantday", "has_resm_onimplantday",
    "has_resp_onorbeforeimplant",
]
lr_results = fn.logrank_all_variables(df, test_vars)
univ_results = fn.univariable_survival_summary(df, test_vars)
univ_results_pub = fn.make_univariable_publication_table(univ_results)

print("Univariable survival screening:")
display(fn.style_univariable_table(univ_results))
print("\nManuscript-ready univariable table:")
display(fn.style_univariable_publication_table(univ_results_pub))
print(f"\nSignificant log-rank tests (p < 0.05): {int(lr_results['Significant'].sum())} / {len(lr_results)}")

In [ ]:
# ============================================================
# 6b. Person-time incidence added to the univariable table
# ============================================================
# Compute per-implant follow-up (censored at INCIDENCE_CUTOFF_DATE, matching
# the PTI incidence calculations) then join implant-years and failure
# incidence rate onto univ_results using the same Variable/Level keys.

_fu = fn.compute_followup_time(
    df,
    study_end=INCIDENCE_CUTOFF_DATE,
    event_col="is_failure",
)[["followup_years", "is_failure"]].copy()
_fu["is_failure"] = pd.to_numeric(_fu["is_failure"], errors="coerce").fillna(0).astype(int)
_fu.index = df.index

_pti_rows = []
for _col in test_vars:
    if _col not in df.columns:
        continue

    # Use the same level coercion as univariable_survival_summary to keep labels aligned.
    _levels_coerced = fn._coerce_univariable_levels(df[_col], _col)
    _tmp = pd.concat([_fu, _levels_coerced.rename("_level")], axis=1).dropna(subset=["_level", "followup_years"])
    _tmp["_level"] = _tmp["_level"].astype(str).str.strip()
    _tmp = _tmp[_tmp["_level"].str.lower().ne("<na>") & _tmp["_level"].ne("")]
    if _tmp.empty or _tmp["_level"].nunique() < 1:
        continue

    _var_rows = univ_results[univ_results["Variable"] == fn._display_variable_name(_col)]
    _levels = [
        str(v).strip()
        for v in _var_rows["Level"].tolist()
        if pd.notna(v) and str(v).strip() != ""
    ]
    if not _levels:
        _levels = fn._reference_order_for_variable(_col, _tmp["_level"])

    for _level in _levels:
        _subset = _tmp[_tmp["_level"] == str(_level)]
        _iy = float(_subset["followup_years"].sum())
        _events = int(_subset["is_failure"].sum())
        _rate = (_events / _iy * 100) if _iy > 0 else float("nan")
        _pti_rows.append(
            {
                "Variable": fn._display_variable_name(_col),
                "Level": str(_level),
                "Implant-years (IY)": round(_iy, 2) if pd.notna(_iy) else float("nan"),
                "Failure incidence rate (per 100 IY)": round(_rate, 2) if pd.notna(_rate) else float("nan"),
            }
        )

if _pti_rows:
    _pti_lookup = pd.DataFrame(_pti_rows).set_index(["Variable", "Level"])
else:
    _pti_lookup = pd.DataFrame(
        columns=[
            "Variable",
            "Level",
            "Implant-years (IY)",
            "Failure incidence rate (per 100 IY)",
        ]
    ).set_index(["Variable", "Level"])

# Merge onto univ_results (header-group rows get NaN -> shown as blank).
_univ_with_pti = univ_results.copy()
_univ_with_pti = _univ_with_pti.join(_pti_lookup, on=["Variable", "Level"], how="left")

# Format new columns as strings so the styler handles them uniformly.
def _fmt_pti(val):
    try:
        f = float(val)
        return "" if pd.isna(f) else f"{f:,.2f}"
    except Exception:
        return ""

_univ_with_pti["Implant-years (IY)"] = _univ_with_pti["Implant-years (IY)"].apply(_fmt_pti)
_univ_with_pti["Failure incidence rate (per 100 IY)"] = _univ_with_pti["Failure incidence rate (per 100 IY)"].apply(_fmt_pti)

print("Univariable analysis with implant-years and failure incidence rate:")
display(fn.style_univariable_table(_univ_with_pti))

# Coverage audit by variable (non-empty levels only).
_audit_rows = []
for _col in test_vars:
    _var_name = fn._display_variable_name(_col)
    _var_table = _univ_with_pti[_univ_with_pti["Variable"] == _var_name].copy()
    _var_table = _var_table[_var_table["Level"].astype(str).str.strip().ne("")]
    if _var_table.empty:
        continue
    _filled = _var_table["Implant-years (IY)"].astype(str).str.strip().ne("").sum()
    _audit_rows.append(
        {
            "Variable": _var_name,
            "Levels in table": int(len(_var_table)),
            "Levels with IY+rate": int(_filled),
        }
    )

if _audit_rows:
    _audit_df = pd.DataFrame(_audit_rows).sort_values("Variable").reset_index(drop=True)
    print("Coverage audit (each level should be filled):")
    display(fn.style_table(_audit_df))

In [ ]:
# ============================================================
# 6c. Export univariable table to Excel in manuscript-style format
# ============================================================
# Output style:
# - title row
# - header row
# - section rows merged across all columns
# - variable names merged vertically in column A across their levels
# - manuscript-like borders, fills, alignment, and widths

import os
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

# Build numeric export version for Excel
_univ_export = univ_results.copy()
_univ_export = _univ_export.join(_pti_lookup, on=["Variable", "Level"], how="left")

out_xlsx = os.path.join(OUTDIR, "univariable_analysis_with_failure_incidence.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    _univ_export.to_excel(writer, sheet_name="Table", index=False)

wb = load_workbook(out_xlsx)
ws = wb["Table"]

# -----------------------------
# Insert title row
# -----------------------------
ws.insert_rows(1)
ws["A1"] = "Univariable analysis with implant-years and failure incidence rate"
ws["A1"].font = Font(bold=True, size=12)
ws["A1"].alignment = Alignment(horizontal="left", vertical="center")

# -----------------------------
# Styles
# -----------------------------
header_fill = PatternFill("solid", fgColor="D9E2F3")
section_fill = PatternFill("solid", fgColor="EDEDED")
white_fill = PatternFill("solid", fgColor="FFFFFF")

thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

header_font = Font(bold=True, size=11)
body_font = Font(size=10)
section_font = Font(bold=True, size=10)
variable_font = Font(size=10)

center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left = Alignment(horizontal="left", vertical="center", wrap_text=True)
right = Alignment(horizontal="right", vertical="center", wrap_text=True)

# -----------------------------
# Header formatting (row 2)
# -----------------------------
for cell in ws[2]:
    cell.font = header_font
    cell.fill = header_fill
    cell.border = border
    cell.alignment = center

# -----------------------------
# Map columns
# -----------------------------
col_idx = {cell.value: i + 1 for i, cell in enumerate(ws[2])}
variable_col = col_idx["Variable"]
level_col = col_idx["Level"]

# -----------------------------
# Helper: identify section rows
# A section row has Variable filled, Level blank,
# and all other cells blank.
# -----------------------------
def is_section_row(ws, row_num, max_col, variable_col, level_col):
    var_val = ws.cell(row=row_num, column=variable_col).value
    level_val = ws.cell(row=row_num, column=level_col).value
    row_values = [ws.cell(row=row_num, column=c).value for c in range(1, max_col + 1)]
    non_empty = [v for v in row_values if v not in (None, "")]
    return (var_val not in (None, "")) and (level_val in (None, "")) and (len(non_empty) == 1)

# -----------------------------
# First pass: base body formatting
# -----------------------------
for row in range(3, ws.max_row + 1):
    section = is_section_row(ws, row, ws.max_column, variable_col, level_col)

    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=row, column=col)
        cell.border = border

        if section:
            cell.font = section_font
            cell.fill = section_fill
            cell.alignment = left
        else:
            cell.font = body_font
            cell.fill = white_fill
            if col in (variable_col, level_col):
                cell.alignment = left
            else:
                cell.alignment = center

# -----------------------------
# Merge section rows across all columns
# -----------------------------
for row in range(3, ws.max_row + 1):
    if is_section_row(ws, row, ws.max_column, variable_col, level_col):
        ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ws.max_column)
        c = ws.cell(row=row, column=1)
        c.font = section_font
        c.fill = section_fill
        c.alignment = left
        c.border = border

# -----------------------------
# Merge Variable cells vertically across repeated level rows
# Example:
# Implant sequence | 1
#                  | 2
#                  | 3+
# becomes one merged Variable cell
# -----------------------------
row = 3
while row <= ws.max_row:
    if is_section_row(ws, row, ws.max_column, variable_col, level_col):
        row += 1
        continue

    var_val = ws.cell(row=row, column=variable_col).value
    level_val = ws.cell(row=row, column=level_col).value

    # Skip blank rows or malformed rows
    if var_val in (None, ""):
        row += 1
        continue

    start_row = row
    end_row = row

    # Continue while next rows belong to same variable block:
    # same Variable value, non-section rows
    next_row = row + 1
    while next_row <= ws.max_row:
        if is_section_row(ws, next_row, ws.max_column, variable_col, level_col):
            break
        next_var = ws.cell(next_row, column=variable_col).value
        if next_var != var_val:
            break
        end_row = next_row
        next_row += 1

    # Merge only if block has more than one row
    if end_row > start_row:
        ws.merge_cells(
            start_row=start_row,
            start_column=variable_col,
            end_row=end_row,
            end_column=variable_col
        )
        c = ws.cell(row=start_row, column=variable_col)
        c.font = variable_font
        c.alignment = left
        c.border = border
        c.fill = white_fill
    else:
        c = ws.cell(row=start_row, column=variable_col)
        c.font = variable_font
        c.alignment = left
        c.border = border
        c.fill = white_fill

    row = end_row + 1

# -----------------------------
# Number formats
# -----------------------------
for row in range(3, ws.max_row + 1):
    if is_section_row(ws, row, ws.max_column, variable_col, level_col):
        continue

    for col_name in ["N", "Events"]:
        if col_name in col_idx:
            ws.cell(row=row, column=col_idx[col_name]).number_format = '#,##0'

    for col_name in ["Success rate (%)", "Implant-years (IY)", "Failure incidence rate (per 100 IY)"]:
        if col_name in col_idx:
            ws.cell(row=row, column=col_idx[col_name]).number_format = '0.00'

# -----------------------------
# Alignment tweaks
# -----------------------------
for row in range(3, ws.max_row + 1):
    if is_section_row(ws, row, ws.max_column, variable_col, level_col):
        continue

    for col_name in col_idx:
        cell = ws.cell(row=row, column=col_idx[col_name])
        if col_name in ["Variable", "Level", "Cox HR (95% CI)"]:
            cell.alignment = left
        else:
            cell.alignment = center

# Re-apply alignment for merged variable anchor cells
row = 3
while row <= ws.max_row:
    if is_section_row(ws, row, ws.max_column, variable_col, level_col):
        row += 1
        continue
    var_val = ws.cell(row=row, column=variable_col).value
    if var_val not in (None, ""):
        ws.cell(row=row, column=variable_col).alignment = left
    row += 1

# -----------------------------
# Column widths
# -----------------------------
widths = {
    "Variable": 26,
    "Level": 22,
    "N": 12,
    "Events": 12,
    "Success rate (%)": 16,
    "Log-rank p-value": 16,
    "Cox HR (95% CI)": 20,
    "Cox p-value": 12,
    "Implant-years (IY)": 18,
    "Failure incidence rate (per 100 IY)": 24,
}
for name, width in widths.items():
    if name in col_idx:
        ws.column_dimensions[get_column_letter(col_idx[name])].width = width

# -----------------------------
# Row heights
# -----------------------------
ws.row_dimensions[1].height = 20
ws.row_dimensions[2].height = 30
for row in range(3, ws.max_row + 1):
    ws.row_dimensions[row].height = 22

# -----------------------------
# Freeze panes / print setup
# -----------------------------
ws.freeze_panes = "A3"
ws.sheet_view.showGridLines = False
ws.print_title_rows = "1:2"
ws.page_setup.fitToWidth = 1
ws.page_setup.fitToHeight = 0
ws.auto_filter.ref = f"A2:{get_column_letter(ws.max_column)}{ws.max_row}"

# -----------------------------
# Ensure title row spans visually
# -----------------------------
ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=ws.max_column)
ws["A1"].alignment = Alignment(horizontal="left", vertical="center")
ws["A1"].font = Font(bold=True, size=12)

wb.save(out_xlsx)
print(f"Saved: {out_xlsx}")

## 7. Cox Model Data Preparation

In [ ]:
# ============================================================
# 7. Prepare the model frame for AG models
# ============================================================
df = fn.prepare_cox_time(df, study_end=CENSOR_DATE)
mf = fn.prepare_model_frame(df)

print(f"Model frame: {len(mf):,} rows")
print(f"Failure events: {int(mf['failure_event'].sum()):,}")

## 8. Andersen–Gill Cox Model – Overall Cohort

This model includes the **full cohort** (all implant failures regardless of timing).
Each patient may contribute multiple implants (AG framework) with robust SE via `cluster(patient_id)`.

In [ ]:
# ============================================================
# 8. AG Model – Overall
# ============================================================
dat_overall = fn.prepare_ag_data(mf)

res_overall, n_overall, ev_overall, c_overall, ph_overall = fn.run_ag_model_r(
    dat_overall, model_label="Andersen–Gill Cox Model – Overall"
)

tbl_overall = fn.result_table(
    res_overall, n_overall, ev_overall, c_overall, ph_overall,
    model_label="AG Model – Overall"
 )
tbl_overall_pub = fn.result_table_publication(
    res_overall, n_overall, ev_overall, c_overall, ph_overall,
    model_label="AG Model – Overall"
)
diag_overall = fn.model_diagnostics_table(
    "AG Model – Overall", n_overall, ev_overall, c_overall, ph_overall
)
display(fn.style_comparison_table(diag_overall))
display(fn.style_result_table(tbl_overall))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_overall_pub))

In [ ]:
# Forest plot – Overall
fig_overall = fn.forest_plot(
    res_overall, n_overall, ev_overall, c_overall,
    model_label="Andersen–Gill Cox Model – Overall",
    save_path=os.path.join(OUTDIR, "forest_AG_overall"),
)
plt.show()

fig_overall_journal = fn.forest_plot_journal(
    res_overall, n_overall, ev_overall, c_overall,
    model_label="Andersen–Gill Cox Model – Overall",
    save_path=os.path.join(OUTDIR, "forest_AG_overall_journal"),
)
plt.show()

## 9. Early / Late Failure Classification

**Definition (from the original analysis):**
- **Early failure:** implant failure within ≤365 days of placement
- **Late failure:** implant failure after >365 days
- **Censored:** no failure observed during follow-up

In [ ]:
# ============================================================
# 9. Classify failures as early / late
# ============================================================
mf = fn.classify_failure_type(mf, threshold_days=EARLY_THRESHOLD_DAYS)

print("Failure type distribution:")
print(mf["failure_type"].value_counts(dropna=False))
print(f"\nEarly threshold: ≤{EARLY_THRESHOLD_DAYS} days")

## 10. Andersen–Gill Cox Model – Early Failures (≤365 days)

For this sub-model, only **early failures** (≤365 days) are treated as events.
Late failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 10. AG Model – Early Failures
# ============================================================
mf_early = mf.copy()
# Re-code: only early failures count as events
mf_early["failure_event"] = np.where(mf_early["failure_type"] == "Early", 1, 0).astype(int)

dat_early = fn.prepare_ag_data(mf_early)

res_early, n_early, ev_early, c_early, ph_early = fn.run_ag_model_r(
    dat_early, model_label="AG Cox Model – Early Failures (≤365 d)"
)

tbl_early = fn.result_table(
    res_early, n_early, ev_early, c_early, ph_early,
    model_label="AG Model – Early Failures"
 )
tbl_early_pub = fn.result_table_publication(
    res_early, n_early, ev_early, c_early, ph_early,
    model_label="AG Model – Early Failures"
)
diag_early = fn.model_diagnostics_table(
    "AG Model – Early Failures", n_early, ev_early, c_early, ph_early
)
display(fn.style_comparison_table(diag_early))
display(fn.style_result_table(tbl_early))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_early_pub))

In [ ]:
# Forest plot – Early
fig_early = fn.forest_plot(
    res_early, n_early, ev_early, c_early,
    model_label="AG Cox Model – Early Failures (≤365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_early"),
)
plt.show()

fig_early_journal = fn.forest_plot_journal(
    res_early, n_early, ev_early, c_early,
    model_label="AG Cox Model – Early Failures (≤365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_early_journal"),
)
plt.show()

## 11. Andersen–Gill Cox Model – Late Failures (>365 days)

For this sub-model, only **late failures** (>365 days) are treated as events.
Early failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 11. AG Model – Late Failures
# ============================================================
mf_late = mf.copy()
# Re-code: only late failures count as events
mf_late["failure_event"] = np.where(mf_late["failure_type"] == "Late", 1, 0).astype(int)

dat_late = fn.prepare_ag_data(mf_late)

res_late, n_late, ev_late, c_late, ph_late = fn.run_ag_model_r(
    dat_late, model_label="AG Cox Model – Late Failures (>365 d)"
)

tbl_late = fn.result_table(
    res_late, n_late, ev_late, c_late, ph_late,
    model_label="AG Model – Late Failures"
 )
tbl_late_pub = fn.result_table_publication(
    res_late, n_late, ev_late, c_late, ph_late,
    model_label="AG Model – Late Failures"
)
diag_late = fn.model_diagnostics_table(
    "AG Model – Late Failures", n_late, ev_late, c_late, ph_late
)
display(fn.style_comparison_table(diag_late))
display(fn.style_result_table(tbl_late))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_late_pub))

In [ ]:
# Forest plot – Late
fig_late = fn.forest_plot(
    res_late, n_late, ev_late, c_late,
    model_label="AG Cox Model – Late Failures (>365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_late"),
)
plt.show()

fig_late_journal = fn.forest_plot_journal(
    res_late, n_late, ev_late, c_late,
    model_label="AG Cox Model – Late Failures (>365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_late_journal"),
)
plt.show()

## 12. Model Comparison Summary and Diagnostics

In [ ]:
# ============================================================
# 12. Side-by-side model comparison
# ============================================================
comparison = pd.concat([diag_overall, diag_early, diag_late], ignore_index=True)
display(fn.style_comparison_table(comparison))

# Merge HR tables side-by-side for the three models
merged = (
    res_overall[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Overall)", "p_fmt": "p (Overall)"})
    .merge(
        res_early[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Early)", "p_fmt": "p (Early)"}),
        on="label", how="outer"
    )
    .merge(
        res_late[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Late)", "p_fmt": "p (Late)"}),
        on="label", how="outer"
    )
)
merged = merged.rename(columns={"label": "Variable"})
display(fn.style_comparison_table(merged))

## 12a. Crude Incidence Rates by Person-Time

This block calculates crude incidence rates by implant-years for overall, early, and late failures, and exports the incidence tables directly to the Excel results workbook.

In [ ]:
# ============================================================
# 12a. Crude incidence rates by implant-years
# ============================================================
study_end_incidence = pd.Timestamp(INCIDENCE_CUTOFF_DATE)
print(f"Incidence cutoff date: {study_end_incidence.date()}")

incidence_df = fn.classify_failure_type(df.copy(), threshold_days=EARLY_THRESHOLD_DAYS)
incidence_df["duedate"] = pd.to_datetime(incidence_df["duedate"], errors="coerce")
incidence_df["failure_date"] = pd.to_datetime(incidence_df["failure_date"], errors="coerce")

implant_group_num = pd.to_numeric(incidence_df["implant_num_surv"], errors="coerce")
incidence_df["implant_group"] = pd.Series(pd.NA, index=incidence_df.index, dtype="string")
incidence_df.loc[implant_group_num.eq(1), "implant_group"] = "1"
incidence_df.loc[implant_group_num.eq(2), "implant_group"] = "2"
incidence_df.loc[implant_group_num.ge(3), "implant_group"] = "3+"
incidence_df = incidence_df.loc[incidence_df["implant_group"].isin(["1", "2", "3+"])].copy()

observed_by_study_end = (
    incidence_df["is_failure"].eq(1)
    & incidence_df["failure_date"].notna()
    & incidence_df["failure_date"].le(study_end_incidence)
)
incidence_df["failure_time_days"] = np.where(
    observed_by_study_end,
    (incidence_df["failure_date"] - incidence_df["duedate"]).dt.days,
    np.nan,
)

overall_followup = fn.compute_followup_time(
    incidence_df.assign(
        is_failure=observed_by_study_end.astype(int),
        failure_date=incidence_df["failure_date"].where(observed_by_study_end),
    ),
    study_end=str(study_end_incidence.date()),
    event_col="is_failure",
)

overall_end = overall_followup["followup_end_date"]
threshold_end = pd.concat(
    [
        incidence_df["duedate"] + pd.to_timedelta(EARLY_THRESHOLD_DAYS, unit="D"),
        pd.Series(study_end_incidence, index=incidence_df.index),
    ],
    axis=1,
).min(axis=1)

early_event = observed_by_study_end & incidence_df["failure_time_days"].le(EARLY_THRESHOLD_DAYS)
early_followup_end = pd.concat([overall_end, threshold_end], axis=1).min(axis=1)
early_followup_days = (early_followup_end - incidence_df["duedate"]).dt.days
early_followup_days = early_followup_days.where(early_followup_days.ge(0))
incidence_df["early_followup_years"] = early_followup_days / 365.25
incidence_df["early_event"] = early_event.astype(int)

late_entry = threshold_end
late_event = observed_by_study_end & incidence_df["failure_time_days"].gt(EARLY_THRESHOLD_DAYS)
late_followup_days = (overall_end - late_entry).dt.days
late_followup_days = late_followup_days.where(late_followup_days.gt(0), 0)
incidence_df["late_followup_years"] = late_followup_days / 365.25
incidence_df["late_event"] = late_event.astype(int)

def summarize_incidence_table(data, group_col, person_time_col, event_col):
    grouped = (
        data.groupby(group_col, dropna=False)
        .agg(
            implants_at_risk=(person_time_col, lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).gt(0).sum())),
            incident_failures=(event_col, "sum"),
            implant_years=(person_time_col, "sum"),
        )
        .reset_index()
        .rename(columns={group_col: "Implant group"})
    )
    grouped.loc[len(grouped)] = [
        "Overall",
        int(grouped["implants_at_risk"].sum()),
        int(grouped["incident_failures"].sum()),
        float(grouped["implant_years"].sum()),
    ]
    grouped["incidence_rate"] = grouped["incident_failures"] / grouped["implant_years"]
    grouped.loc[grouped["implant_years"].le(0), "incidence_rate"] = np.nan
    grouped["rate_per_100_implant_years"] = grouped["incidence_rate"] * 100
    return grouped

incidence_overall = summarize_incidence_table(
    overall_followup.assign(implant_group=incidence_df["implant_group"]),
    group_col="implant_group",
    person_time_col="followup_years",
    event_col="is_failure",
)
incidence_early = summarize_incidence_table(
    incidence_df,
    group_col="implant_group",
    person_time_col="early_followup_years",
    event_col="early_event",
)
incidence_late = summarize_incidence_table(
    incidence_df,
    group_col="implant_group",
    person_time_col="late_followup_years",
    event_col="late_event",
)

incidence_formats = {
    "implant_years": "{:,.2f}",
    "incidence_rate": "{:,.4f}",
    "rate_per_100_implant_years": "{:,.2f}",
}

display(fn.style_table(
    incidence_overall,
    caption="Overall crude incidence by implant group",
    formats=incidence_formats,
))
display(fn.style_table(
    incidence_early,
    caption="Early crude incidence by implant group (<=365 days)",
    formats=incidence_formats,
))
display(fn.style_table(
    incidence_late,
    caption="Late crude incidence by implant group (>365 days)",
    formats=incidence_formats,
))

overall_total_py = float(incidence_overall.loc[incidence_overall["Implant group"].eq("Overall"), "implant_years"].iloc[0])
early_total_py = float(incidence_early.loc[incidence_early["Implant group"].eq("Overall"), "implant_years"].iloc[0])
late_total_py = float(incidence_late.loc[incidence_late["Implant group"].eq("Overall"), "implant_years"].iloc[0])

incidence_checks = pd.DataFrame([
    {
        "Definition": "Overall",
        "Events": int(incidence_overall.loc[incidence_overall["Implant group"].eq("Overall"), "incident_failures"].iloc[0]),
        "Implant-years": overall_total_py,
    },
    {
        "Definition": "Early",
        "Events": int(incidence_early.loc[incidence_early["Implant group"].eq("Overall"), "incident_failures"].iloc[0]),
        "Implant-years": early_total_py,
    },
    {
        "Definition": "Late",
        "Events": int(incidence_late.loc[incidence_late["Implant group"].eq("Overall"), "incident_failures"].iloc[0]),
        "Implant-years": late_total_py,
    },
    {
        "Definition": "Early + Late minus Overall",
        "Events": int(
            incidence_early.loc[incidence_early["Implant group"].eq("Overall"), "incident_failures"].iloc[0]
            + incidence_late.loc[incidence_late["Implant group"].eq("Overall"), "incident_failures"].iloc[0]
            - incidence_overall.loc[incidence_overall["Implant group"].eq("Overall"), "incident_failures"].iloc[0]
        ),
        "Implant-years": early_total_py + late_total_py - overall_total_py,
    },
])
print("Incidence totals:")
display(incidence_checks)

pti_excel_path = os.path.join(OUTDIR, "AG_model_results.xlsx")
writer_kwargs = (
    {"engine": "openpyxl", "mode": "a", "if_sheet_exists": "replace"}
    if os.path.exists(pti_excel_path)
    else {"engine": "openpyxl", "mode": "w"}
)
try:
    with pd.ExcelWriter(pti_excel_path, **writer_kwargs) as writer:
        incidence_overall.to_excel(writer, sheet_name="Incidence_overall", index=False)
        incidence_early.to_excel(writer, sheet_name="Incidence_early", index=False)
        incidence_late.to_excel(writer, sheet_name="Incidence_late", index=False)
        incidence_checks.to_excel(writer, sheet_name="Incidence_checks", index=False)
    print(f"Incidence tables exported to: {pti_excel_path}")
except PermissionError:
    print("Could not write incidence tables because AG_model_results.xlsx is open.")
    print("Close the file and re-run this cell to export.")

In [ ]:

# ============================================================
# 12b. Person-time incidence by restoration type
# ============================================================
# Restoration groups:
#   rehabilitation_type == "fixed"            → "Fixed"
#   rehabilitation_type == "denture_inferred" → "Denture"
#   rehabilitation_type == "unknown" / other  → "None"
# incidence_df and overall_followup share the same index (both
# come from cell 12a), so alignment is guaranteed.

_rehab_raw = incidence_df["rehabilitation_type"].astype(str).str.strip().str.lower()
_restoration_group = _rehab_raw.map({
    "fixed": "Fixed",
    "denture_inferred": "Denture",
}).fillna("None")

_rest_fu = overall_followup[["followup_years", "is_failure"]].copy()
_rest_fu["restoration_group"] = _restoration_group.values

_rest_agg = (
    _rest_fu.groupby("restoration_group", dropna=False)
    .agg(
        N=("followup_years", "size"),
        Failures=("is_failure", "sum"),
        Implant_years=("followup_years", "sum"),
    )
    .reindex(["Denture", "Fixed", "None"])
    .fillna(0)
    .reset_index()
)
_rest_agg["Rate_per_100_IY"] = (_rest_agg["Failures"] / _rest_agg["Implant_years"] * 100).where(
    _rest_agg["Implant_years"] > 0
)
_rest_agg["N"] = _rest_agg["N"].astype(int)
_rest_agg["Failures"] = _rest_agg["Failures"].astype(int)

display(fn.style_table(
    _rest_agg.rename(columns={
        "restoration_group": "Restoration type",
        "Implant_years": "Implant-years",
        "Rate_per_100_IY": "Rate per 100 implant-years",
    }),
    caption="Person-time incidence by restoration type",
    formats={"Implant-years": "{:,.2f}", "Rate per 100 implant-years": "{:.2f}"},
))

# --- Bar chart -------------------------------------------------------
_rest_colors = {
    "Denture": fn.WONG["orange"],
    "Fixed":   fn.WONG["blue"],
    "None":    fn.WONG["vermil"],
}

fn._apply_jcp_style()
fig_rest_incidence, ax_rest = plt.subplots(figsize=(7, 4.5), constrained_layout=True)

_x = np.arange(len(_rest_agg))
_bar_width = 0.55
_bars = ax_rest.bar(
    _x,
    _rest_agg["Rate_per_100_IY"].fillna(0),
    _bar_width,
    color=[_rest_colors.get(g, "#555555") for g in _rest_agg["restoration_group"]],
    edgecolor="black",
    linewidth=0.5,
)

# Value labels above each bar
_max_rate = _rest_agg["Rate_per_100_IY"].max(skipna=True)
for _bar, _rate in zip(_bars, _rest_agg["Rate_per_100_IY"]):
    if pd.notna(_rate):
        ax_rest.text(
            _bar.get_x() + _bar.get_width() / 2,
            _bar.get_height() + 0.02 * _max_rate,
            f"{_rate:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=fn.FIG_DEFAULTS["title_color"],
        )

# X-axis labels with n= counts
_xlabels = [
    f"{row['restoration_group']}\nn={row['N']:,}"
    for _, row in _rest_agg.iterrows()
]
ax_rest.set_xticks(_x)
ax_rest.set_xticklabels(_xlabels)
ax_rest.set_ylabel("Rate per 100 implant-years")
ax_rest.set_xlabel("Restoration type")
ax_rest.set_title(
    "Person-Time Incidence by Restoration Type",
    color=fn.FIG_DEFAULTS["title_color"],
    pad=10,
)
ax_rest.yaxis.grid(True, linestyle="--", color=fn.FIG_DEFAULTS["grid_color"], alpha=0.7)
ax_rest.set_axisbelow(True)
for _spine in ["top", "right"]:
    ax_rest.spines[_spine].set_visible(False)

_save_rest = os.path.join(OUTDIR, "incidence_by_restoration_type")
for _ext in ("png", "pdf"):
    fig_rest_incidence.savefig(f"{_save_rest}.{_ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
print(f"  Saved: {_save_rest}.png / .pdf")
plt.show()


In [ ]:
# ============================================================
# 12c. Failure incidence rate by implant index (1, 2, 3+) – bar chart
# ============================================================

from matplotlib.ticker import MaxNLocator

_inc_plot = (
    incidence_overall
    .loc[incidence_overall["Implant group"].isin(["1", "2", "3+"])]
    .copy()
    .reset_index(drop=True)
)
_inc_plot["Implant group"] = pd.Categorical(
    _inc_plot["Implant group"], categories=["1", "2", "3+"], ordered=True
)
_inc_plot = _inc_plot.sort_values("Implant group").reset_index(drop=True)

_inc_colors = {
    "1": fn.WONG["blue"],
    "2": fn.WONG["orange"],
    "3+": fn.WONG["green"],
}

_seq_labels = {
    "1": "1st implant",
    "2": "2nd implant",
    "3+": "3rd+ implant",
}

# X labels: only implant sequence + n
_xlabels = [
    (
        f"{_seq_labels.get(str(row['Implant group']), str(row['Implant group']))}\n"
        f"n={int(row['implants_at_risk']):,}"
    )
    for _, row in _inc_plot.iterrows()
]

# Figure legend text for JCP
_rates = {
    str(row["Implant group"]): float(row["rate_per_100_implant_years"])
    for _, row in _inc_plot.iterrows()
}
_ns = {
    str(row["Implant group"]): int(row["implants_at_risk"])
    for _, row in _inc_plot.iterrows()
}
_pys = {
    str(row["Implant group"]): float(row["implant_years"])
    for _, row in _inc_plot.iterrows()
}

figure_legend = (
    "Figure X. Incidence rate of implant failure by procedure sequence. "
    "Bar plot showing implant failure incidence rates according to procedure sequence "
    "(1st implant, 2nd implant, and 3rd+ implant). "
    "Incidence rates are expressed as events per 100 implant-years. "
    f"Rates were {_rates['1']:.2f}, {_rates['2']:.2f}, and {_rates['3+']:.2f} events per 100 implant-years, respectively. "
    f"Sample sizes were n={_ns['1']:,}, n={_ns['2']:,}, and n={_ns['3+']:,}, "
    f"with corresponding person-years (PY) of {_pys['1']:,.1f}, {_pys['2']:,.1f}, and {_pys['3+']:,.1f}."
)

fn._apply_jcp_style()
fig_inc_seq, ax_inc_seq = plt.subplots(figsize=(8.4, 5.2), constrained_layout=True)
ax_inc_seq.set_facecolor("white")

_x = np.arange(len(_inc_plot))
_bars = ax_inc_seq.bar(
    _x,
    _inc_plot["rate_per_100_implant_years"],
    width=0.58,
    color=[_inc_colors.get(g, "#555555") for g in _inc_plot["Implant group"].astype(str)],
    edgecolor=fn.FIG_DEFAULTS["border_color"],
    linewidth=0.6,
)

_y_max = float(_inc_plot["rate_per_100_implant_years"].max()) if len(_inc_plot) else 0.0
_label_offset = 0.04 * _y_max if _y_max > 0 else 0.05

for _bar, _rate in zip(_bars, _inc_plot["rate_per_100_implant_years"]):
    ax_inc_seq.text(
        _bar.get_x() + _bar.get_width() / 2,
        _bar.get_height() + _label_offset,
        f"{_rate:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=fn.FIG_DEFAULTS["title_color"],
    )

ax_inc_seq.set_xticks(_x)
ax_inc_seq.set_xticklabels(_xlabels, ha="center")
ax_inc_seq.tick_params(axis="x", labelsize=9, pad=8)
ax_inc_seq.tick_params(axis="y", labelsize=9)

ax_inc_seq.set_ylabel(
    "Incidence rate of implant failure (events per 100 implant-years)",
    fontsize=10.5,
)
ax_inc_seq.set_title(
    "Incidence Rate of Implant Failure by Procedure Sequence",
    color=fn.FIG_DEFAULTS["title_color"],
    pad=10,
    fontsize=13,
    fontweight="bold",
)

ax_inc_seq.set_ylim(0, _y_max * 1.22 if _y_max > 0 else 1)
ax_inc_seq.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax_inc_seq.yaxis.grid(
    True,
    linestyle="--",
    color=fn.FIG_DEFAULTS["grid_color"],
    alpha=0.45,
)
ax_inc_seq.set_axisbelow(True)
ax_inc_seq.margins(x=0.08)

for _spine in ["top", "right"]:
    ax_inc_seq.spines[_spine].set_visible(False)

_save_inc_seq = os.path.join(OUTDIR, "failure_incidence_rate_by_procedure_sequence")
for _ext in ("png", "pdf"):
    fig_inc_seq.savefig(
        f"{_save_inc_seq}.{_ext}",
        dpi=max(fn.FIG_DEFAULTS["dpi"], 600),
        bbox_inches="tight",
    )

print(f"  Saved: {_save_inc_seq}.png / .pdf")
plt.show()

# Print figure legend below the plot as notebook output
print(figure_legend)

## 13. Export Results

In [ ]:
# ============================================================
# 13. Save tables to Excel
# ============================================================
excel_path = os.path.join(OUTDIR, "AG_model_results.xlsx")

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    tbl_overall.to_excel(writer, sheet_name="Overall", index=False)
    tbl_early.to_excel(writer, sheet_name="Early", index=False)
    tbl_late.to_excel(writer, sheet_name="Late", index=False)
    merged.to_excel(writer, sheet_name="Comparison", index=False)
    comparison.to_excel(writer, sheet_name="Summary", index=False)
    univ_results.to_excel(writer, sheet_name="Univariable", index=False)
    lr_results.to_excel(writer, sheet_name="LogRank", index=False)

    tbl_overall_pub.to_excel(writer, sheet_name="Overall_pub", index=False)
    tbl_early_pub.to_excel(writer, sheet_name="Early_pub", index=False)
    tbl_late_pub.to_excel(writer, sheet_name="Late_pub", index=False)
    univ_results_pub.to_excel(writer, sheet_name="Univariable_pub", index=False)

print(f"Results saved to: {excel_path}")
print(f"Forest plots saved to: {OUTDIR}")
print("\nDone. All outputs ready for JCP submission.")